# 🐻‍❄️ Polars — Complete Learning & Practice Workbook

## What is Polars?
**Polars** is a blazing-fast DataFrame library for Python, written in **Rust**.  
It is designed as a modern replacement for **pandas**, fixing its core performance and usability problems.

---

## Why Polars over pandas?

| Feature | pandas | Polars |
|---|---|---|
| Speed | Slow on large data | 5–100× faster |
| Memory | High (copies everywhere) | Low (zero-copy Arrow format) |
| Parallelism | Single-threaded | Multi-threaded by default |
| Lazy evaluation | ❌ | ✅ — builds query plans |
| Null handling | NaN / None confusion | Clean `null` type |
| API consistency | Mixed (index-based) | Consistent expression API |

---

## Core concepts covered in this workbook

| # | Module | Topic |
|---|--------|-------|
| 1 | Setup | Install & import, version check |
| 2 | Series | The 1D building block |
| 3 | DataFrame | Creating, inspecting, selecting |
| 4 | Expressions | The `pl.col()` expression system |
| 5 | Filtering & Sorting | Row-level operations |
| 6 | Adding & Transforming Columns | `with_columns`, `map_elements` |
| 7 | GroupBy & Aggregations | Split-apply-combine |
| 8 | Joins | Combining DataFrames |
| 9 | Lazy API | Query planning & optimisation |
| 10 | I/O | CSV, JSON, Parquet read/write |
| 11 | Missing Data | Handling nulls |
| 12 | Real-world Example | Full EDA pipeline |


---
## Module 1 — Setup & Installation

Polars uses the **Apache Arrow** columnar memory format internally.  
This means data is stored column-by-column (not row-by-row like pandas), which is ideal for analytical queries that typically operate on one or two columns at a time.


In [1]:
import subprocess, sys

try:
    import polars as pl
    print(f"✅ Polars already installed: v{pl.__version__}")
except ImportError:
    print("Installing polars...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "polars", "-q"])
    import polars as pl
    print(f"✅ Polars installed: v{pl.__version__}")

# Polars uses Apache Arrow under the hood — let's confirm
print(f"   Arrow-backed: yes (pyarrow or polars native Arrow)")
print(f"   Available dtypes: {[str(d) for d in [pl.Int64, pl.Float64, pl.Utf8, pl.Boolean, pl.Date, pl.Datetime]]}")


✅ Polars already installed: v1.39.3
   Arrow-backed: yes (pyarrow or polars native Arrow)
   Available dtypes: ['Int64', 'Float64', 'String', 'Boolean', 'Date', 'Datetime']


---
## Module 2 — Series: The 1D Building Block

A **Series** is a single named column of typed data — the 1D equivalent of a DataFrame column.  
Every value in a Series has the **same dtype** (unlike Python lists). This is what enables vectorised operations without Python loops.

Key properties:
- `name` — the column label
- `dtype` — the data type (Int64, Float64, Utf8, Boolean, Date, etc.)
- `len()` — number of elements
- Supports vectorised arithmetic, comparisons, and string operations natively


In [2]:
import polars as pl

# --- Creating Series ---
scores = pl.Series("scores", [88, 95, 72, 61, 99, 84])
names  = pl.Series("names",  ["Alice", "Bob", "Carol", "Dave", "Eve", "Frank"])
passed = pl.Series("passed", [True, True, False, False, True, True])

print("=== Basic Series info ===")
print(f"Name : {scores.name}")
print(f"Dtype: {scores.dtype}")
print(f"Len  : {scores.len()}")
print(f"Data : {scores.to_list()}\n")

# --- Vectorised arithmetic (no loops!) ---
print("=== Arithmetic (vectorised) ===")
print("scores + 5    :", (scores + 5).to_list())
print("scores * 1.1  :", (scores * 1.1).round(1).to_list())
print("scores > 80   :", (scores > 80).to_list())

# --- Descriptive statistics ---
print("\n=== Statistics ===")
print(f"mean : {scores.mean():.1f}")
print(f"std  : {scores.std():.1f}")
print(f"min  : {scores.min()}")
print(f"max  : {scores.max()}")
print(f"median: {scores.median()}")

# --- String operations via .str accessor ---
print("\n=== String operations (.str accessor) ===")
print("uppercase  :", names.str.to_uppercase().to_list())
print("starts A   :", names.str.starts_with("A").to_list())
print("length     :", names.str.len_chars().to_list())

# --- Null handling ---
s_with_nulls = pl.Series("vals", [1, None, 3, None, 5])
print("\n=== Nulls ===")
print("raw          :", s_with_nulls.to_list())
print("null_count   :", s_with_nulls.null_count())
print("fill_null(0) :", s_with_nulls.fill_null(0).to_list())
print("drop_nulls   :", s_with_nulls.drop_nulls().to_list())


=== Basic Series info ===
Name : scores
Dtype: Int64
Len  : 6
Data : [88, 95, 72, 61, 99, 84]

=== Arithmetic (vectorised) ===
scores + 5    : [93, 100, 77, 66, 104, 89]
scores * 1.1  : [96.8, 104.5, 79.2, 67.1, 108.9, 92.4]
scores > 80   : [True, True, False, False, True, True]

=== Statistics ===
mean : 83.2
std  : 14.4
min  : 61
max  : 99
median: 86.0

=== String operations (.str accessor) ===
uppercase  : ['ALICE', 'BOB', 'CAROL', 'DAVE', 'EVE', 'FRANK']
starts A   : [True, False, False, False, False, False]
length     : [5, 3, 5, 4, 3, 5]

=== Nulls ===
raw          : [1, None, 3, None, 5]
null_count   : 2
fill_null(0) : [1, 0, 3, 0, 5]
drop_nulls   : [1, 3, 5]


---
## Module 3 — DataFrame: Creating & Inspecting

A **DataFrame** is a 2D table of typed columns (Series). All columns must have the same length.

Polars DataFrames have **no index** — unlike pandas, there is no special row label. Rows are accessed by position or by filtering expressions. This keeps the API simpler and faster.

**Ways to create a DataFrame:**
- From a Python `dict` of lists
- From a list of `dicts` (records)
- From a CSV / Parquet / JSON file
- From a `pandas` DataFrame (via `pl.from_pandas()`)


In [3]:
import polars as pl
from datetime import date

# --- Create a realistic sample DataFrame ---
df = pl.DataFrame({
    "employee_id":  [101, 102, 103, 104, 105, 106, 107],
    "name":         ["Alice", "Bob", "Carol", "Dave", "Eve", "Frank", "Grace"],
    "department":   ["Eng", "Eng", "HR", "HR", "Eng", "Finance", "Finance"],
    "salary":       [95000, 82000, 67000, 71000, 110000, 88000, 92000],
    "years":        [5, 3, 8, 2, 10, 6, 4],
    "remote":       [True, False, True, True, False, False, True],
    "start_date":   [
        date(2019, 3, 1), date(2021, 7, 15), date(2016, 1, 10),
        date(2022, 4, 5), date(2014, 9, 20), date(2018, 11, 30), date(2020, 6, 8)
    ],
})

print("=== DataFrame overview ===")
print(df)

print("\n=== Shape (rows, cols) ===")
print(df.shape)

print("\n=== Column names & dtypes ===")
print(df.schema)

print("\n=== Summary statistics ===")
print(df.describe())

print("\n=== First 3 rows ===")
print(df.head(3))

print("\n=== Last 2 rows ===")
print(df.tail(2))


=== DataFrame overview ===
shape: (7, 7)
┌─────────────┬───────┬────────────┬────────┬───────┬────────┬────────────┐
│ employee_id ┆ name  ┆ department ┆ salary ┆ years ┆ remote ┆ start_date │
│ ---         ┆ ---   ┆ ---        ┆ ---    ┆ ---   ┆ ---    ┆ ---        │
│ i64         ┆ str   ┆ str        ┆ i64    ┆ i64   ┆ bool   ┆ date       │
╞═════════════╪═══════╪════════════╪════════╪═══════╪════════╪════════════╡
│ 101         ┆ Alice ┆ Eng        ┆ 95000  ┆ 5     ┆ true   ┆ 2019-03-01 │
│ 102         ┆ Bob   ┆ Eng        ┆ 82000  ┆ 3     ┆ false  ┆ 2021-07-15 │
│ 103         ┆ Carol ┆ HR         ┆ 67000  ┆ 8     ┆ true   ┆ 2016-01-10 │
│ 104         ┆ Dave  ┆ HR         ┆ 71000  ┆ 2     ┆ true   ┆ 2022-04-05 │
│ 105         ┆ Eve   ┆ Eng        ┆ 110000 ┆ 10    ┆ false  ┆ 2014-09-20 │
│ 106         ┆ Frank ┆ Finance    ┆ 88000  ┆ 6     ┆ false  ┆ 2018-11-30 │
│ 107         ┆ Grace ┆ Finance    ┆ 92000  ┆ 4     ┆ true   ┆ 2020-06-08 │
└─────────────┴───────┴────────────┴────────┴──

---
## Module 4 — Selecting Columns & The Expression System (`pl.col`)

### Theory: Expressions
This is **the most important concept in Polars**.

An **expression** (`pl.col("name")`) is a *lazy description* of an operation — it doesn't execute immediately. Expressions are composed together and only evaluated when passed to `.select()`, `.filter()`, `.with_columns()`, etc.

This allows Polars to:
- **Optimise** the query before running it
- **Parallelise** independent expressions automatically

```python
# pandas style (eager, index-based, verbose)
df["salary"] * 1.1

# Polars expression style (composable, optimisable)
pl.col("salary") * 1.1
```

**Column selection shortcuts:**
- `pl.col("a")` — single column
- `pl.col("a", "b", "c")` — multiple by name
- `pl.col(pl.Int64)` — all columns of a dtype
- `pl.all()` — every column
- `pl.exclude("a")` — all except `a`


In [4]:
import polars as pl
from datetime import date

df = pl.DataFrame({
    "employee_id":  [101, 102, 103, 104, 105, 106, 107],
    "name":         ["Alice", "Bob", "Carol", "Dave", "Eve", "Frank", "Grace"],
    "department":   ["Eng", "Eng", "HR", "HR", "Eng", "Finance", "Finance"],
    "salary":       [95000, 82000, 67000, 71000, 110000, 88000, 92000],
    "years":        [5, 3, 8, 2, 10, 6, 4],
    "remote":       [True, False, True, True, False, False, True],
    "start_date":   [date(2019,3,1), date(2021,7,15), date(2016,1,10),
                     date(2022,4,5), date(2014,9,20), date(2018,11,30), date(2020,6,8)],
})

# --- select: choose and transform columns ---
print("=== select: pick columns ===")
print(df.select("name", "salary"))

print("\n=== select with expression: salary in thousands ===")
print(df.select(
    pl.col("name"),
    (pl.col("salary") / 1000).round(1).alias("salary_k"),
))

print("\n=== select all numeric columns ===")
print(df.select(pl.col(pl.Int64)))

print("\n=== select all except employee_id ===")
print(df.select(pl.exclude("employee_id")))

# --- Computed expressions ---
print("\n=== Computed column: monthly salary ===")
print(df.select(
    pl.col("name"),
    pl.col("salary"),
    (pl.col("salary") / 12).round(2).alias("monthly_salary"),
    (pl.col("salary") > pl.col("salary").mean()).alias("above_average"),
))


=== select: pick columns ===
shape: (7, 2)
┌───────┬────────┐
│ name  ┆ salary │
│ ---   ┆ ---    │
│ str   ┆ i64    │
╞═══════╪════════╡
│ Alice ┆ 95000  │
│ Bob   ┆ 82000  │
│ Carol ┆ 67000  │
│ Dave  ┆ 71000  │
│ Eve   ┆ 110000 │
│ Frank ┆ 88000  │
│ Grace ┆ 92000  │
└───────┴────────┘

=== select with expression: salary in thousands ===
shape: (7, 2)
┌───────┬──────────┐
│ name  ┆ salary_k │
│ ---   ┆ ---      │
│ str   ┆ f64      │
╞═══════╪══════════╡
│ Alice ┆ 95.0     │
│ Bob   ┆ 82.0     │
│ Carol ┆ 67.0     │
│ Dave  ┆ 71.0     │
│ Eve   ┆ 110.0    │
│ Frank ┆ 88.0     │
│ Grace ┆ 92.0     │
└───────┴──────────┘

=== select all numeric columns ===
shape: (7, 3)
┌─────────────┬────────┬───────┐
│ employee_id ┆ salary ┆ years │
│ ---         ┆ ---    ┆ ---   │
│ i64         ┆ i64    ┆ i64   │
╞═════════════╪════════╪═══════╡
│ 101         ┆ 95000  ┆ 5     │
│ 102         ┆ 82000  ┆ 3     │
│ 103         ┆ 67000  ┆ 8     │
│ 104         ┆ 71000  ┆ 2     │
│ 105         ┆ 110000 

---
## Module 5 — Filtering & Sorting

### Theory
**Filtering** in Polars uses `.filter(expression)` where the expression evaluates to a Boolean Series.

Conditions can be chained using:
- `&` → AND
- `|` → OR  
- `~` → NOT

**Sorting** uses `.sort("column")` or `.sort(["col1", "col2"], descending=[True, False])` for multi-column sort.

> ⚠️ **Important:** In Polars, use `&` and `|` (not `and`/`or`) between conditions, and always wrap each condition in parentheses.


In [ ]:
import polars as pl
from datetime import date

df = pl.DataFrame({
    "name":       ["Alice", "Bob", "Carol", "Dave", "Eve", "Frank", "Grace"],
    "department": ["Eng", "Eng", "HR", "HR", "Eng", "Finance", "Finance"],
    "salary":     [95000, 82000, 67000, 71000, 110000, 88000, 92000],
    "years":      [5, 3, 8, 2, 10, 6, 4],
    "remote":     [True, False, True, True, False, False, True],
})

# --- Simple filter ---
print("=== Salary > 85,000 ===")
print(df.filter(pl.col("salary") > 85_000))

# --- Multi-condition filter: AND ---
print("\n=== Eng department AND salary > 90k ===")
print(df.filter(
    (pl.col("department") == "Eng") & (pl.col("salary") > 90_000)
))

# --- OR condition ---
print("\n=== Finance OR remote workers ===")
print(df.filter(
    (pl.col("department") == "Finance") | (pl.col("remote") == True)
))

# --- Filter with string methods ---
print("\n=== Names starting with vowels ===")
print(df.filter(pl.col("name").str.starts_with(("A", "E"))))

# --- Filter using .is_in() ---
print("\n=== HR or Finance departments ===")
print(df.filter(pl.col("department").is_in(["HR", "Finance"])))

# --- Sorting ---
print("\n=== Sorted by salary descending ===")
print(df.sort("salary", descending=True))

print("\n=== Multi-sort: department asc, salary desc ===")
print(df.sort(["department", "salary"], descending=[False, True]))


---
## Module 6 — Adding & Transforming Columns (`with_columns`)

### Theory
`.with_columns(...)` adds new columns or **replaces** existing ones. It always returns a new DataFrame (Polars is **immutable** — operations never modify in place).

You pass one or more expressions. Each expression must:
- Evaluate to the same length as the DataFrame, OR
- Be a scalar (which gets broadcast to all rows)

Use `.alias("new_name")` to name the result. If you use the same name as an existing column, it replaces it.

**`pl.when().then().otherwise()`** — the Polars equivalent of `np.where` or SQL `CASE WHEN`.


In [ ]:
import polars as pl

df = pl.DataFrame({
    "name":       ["Alice", "Bob", "Carol", "Dave", "Eve", "Frank", "Grace"],
    "department": ["Eng", "Eng", "HR", "HR", "Eng", "Finance", "Finance"],
    "salary":     [95000, 82000, 67000, 71000, 110000, 88000, 92000],
    "years":      [5, 3, 8, 2, 10, 6, 4],
})

# --- Add derived columns ---
print("=== Adding columns with with_columns ===")
df2 = df.with_columns(
    (pl.col("salary") / 12).round(2).alias("monthly_salary"),
    (pl.col("salary") / pl.col("years")).round(0).alias("salary_per_year_exp"),
    pl.col("name").str.to_uppercase().alias("name_upper"),
)
print(df2)

# --- pl.when / then / otherwise (like CASE WHEN in SQL) ---
print("\n=== pl.when().then().otherwise() ===")
df3 = df.with_columns(
    pl.when(pl.col("salary") >= 90_000)
      .then(pl.lit("Senior"))
      .when(pl.col("salary") >= 75_000)
      .then(pl.lit("Mid"))
      .otherwise(pl.lit("Junior"))
      .alias("level"),

    pl.when(pl.col("years") > 5)
      .then(pl.lit("Veteran"))
      .otherwise(pl.lit("New"))
      .alias("tenure_band"),
)
print(df3.select("name", "salary", "years", "level", "tenure_band"))

# --- Rename columns ---
print("\n=== Rename ===")
print(df.rename({"salary": "annual_pay", "years": "experience_yrs"}))

# --- Cast dtype ---
print("\n=== Cast salary to Float32 ===")
print(df.with_columns(pl.col("salary").cast(pl.Float32)))


---
## Module 7 — GroupBy & Aggregations

### Theory
**GroupBy** splits the DataFrame into groups by one or more columns, applies an aggregation to each group, and combines the results — the classic **Split → Apply → Combine** pattern.

In Polars, this is:
```python
df.group_by("column").agg(pl.col("other").mean())
```

Common aggregation functions:
| Function | Description |
|---|---|
| `.mean()` | Average |
| `.sum()` | Total |
| `.min()` / `.max()` | Range |
| `.count()` | Row count |
| `.std()` | Standard deviation |
| `.first()` / `.last()` | First / last value in group |
| `.n_unique()` | Distinct value count |
| `.list()` | Collect all values into a list |

> ⚠️ Group order is **not guaranteed** in Polars by default. Use `.sort()` after `group_by` if order matters.  
> Use `group_by_dynamic()` for time-series resampling.


In [ ]:
import polars as pl

df = pl.DataFrame({
    "name":       ["Alice", "Bob", "Carol", "Dave", "Eve", "Frank", "Grace"],
    "department": ["Eng", "Eng", "HR", "HR", "Eng", "Finance", "Finance"],
    "salary":     [95000, 82000, 67000, 71000, 110000, 88000, 92000],
    "years":      [5, 3, 8, 2, 10, 6, 4],
    "remote":     [True, False, True, True, False, False, True],
})

# --- Basic group_by + single agg ---
print("=== Mean salary by department ===")
print(
    df.group_by("department")
      .agg(pl.col("salary").mean().round(0).alias("avg_salary"))
      .sort("avg_salary", descending=True)
)

# --- Multiple aggregations at once ---
print("\n=== Full summary by department ===")
print(
    df.group_by("department")
      .agg(
          pl.col("salary").mean().round(0).alias("avg_salary"),
          pl.col("salary").max().alias("max_salary"),
          pl.col("salary").min().alias("min_salary"),
          pl.col("years").mean().round(1).alias("avg_years"),
          pl.len().alias("headcount"),
          pl.col("remote").sum().alias("remote_count"),
      )
      .sort("department")
)

# --- Group by multiple columns ---
print("\n=== Group by department + remote ===")
print(
    df.group_by("department", "remote")
      .agg(
          pl.col("salary").mean().round(0).alias("avg_salary"),
          pl.len().alias("count"),
      )
      .sort(["department", "remote"])
)

# --- Collect values into a list per group ---
print("\n=== Names per department (as list) ===")
print(
    df.group_by("department")
      .agg(pl.col("name").sort().alias("members"))
      .sort("department")
)


---
## Module 8 — Joins

### Theory
Polars supports all standard SQL join types with `.join()`:

| Join type | Keeps | SQL equivalent |
|---|---|---|
| `"inner"` | Only matching rows from both sides | `INNER JOIN` |
| `"left"` | All rows from left, nulls for missing right | `LEFT JOIN` |
| `"full"` | All rows from both, nulls for non-matches | `FULL OUTER JOIN` |
| `"semi"` | Left rows where a match exists (no right cols) | `WHERE EXISTS` |
| `"anti"` | Left rows where **no** match exists | `WHERE NOT EXISTS` |
| `"cross"` | Cartesian product of all rows | `CROSS JOIN` |

Syntax:
```python
df_left.join(df_right, on="key_column", how="inner")
# or for different column names:
df_left.join(df_right, left_on="id", right_on="emp_id", how="left")
```


In [ ]:
import polars as pl

employees = pl.DataFrame({
    "emp_id":     [101, 102, 103, 104, 105],
    "name":       ["Alice", "Bob", "Carol", "Dave", "Eve"],
    "dept_id":    [1, 1, 2, 2, 1],
})

departments = pl.DataFrame({
    "dept_id":    [1, 2, 3],
    "dept_name":  ["Engineering", "HR", "Finance"],
    "budget_k":   [500, 200, 350],
})

projects = pl.DataFrame({
    "project":    ["Alpha", "Beta", "Gamma"],
    "lead_id":    [101, 103, 999],   # 999 doesn't exist in employees
})

# --- Inner join ---
print("=== INNER JOIN: employees + departments ===")
print(employees.join(departments, on="dept_id", how="inner"))

# --- Left join ---
print("\n=== LEFT JOIN: projects keep all, even if no matching employee ===")
print(
    projects.join(employees, left_on="lead_id", right_on="emp_id", how="left")
)

# --- Semi join: which employees are project leads? ---
print("\n=== SEMI JOIN: employees who lead a project ===")
print(employees.join(projects, left_on="emp_id", right_on="lead_id", how="semi"))

# --- Anti join: employees NOT leading any project ---
print("\n=== ANTI JOIN: employees with no project ===")
print(employees.join(projects, left_on="emp_id", right_on="lead_id", how="anti"))

# --- Chained joins ---
print("\n=== Chained: employees + depts + project info ===")
print(
    employees
    .join(departments, on="dept_id", how="left")
    .join(projects, left_on="emp_id", right_on="lead_id", how="left")
    .select("name", "dept_name", "project", "budget_k")
)


---
## Module 9 — The Lazy API: Query Planning & Optimisation

### Theory — This is where Polars really shines

The **Lazy API** lets you build up a query as a **plan** without executing it.  
When you call `.collect()`, Polars compiles the plan and applies optimisations:

- **Predicate pushdown** — filters are moved as early as possible (scan less data)
- **Projection pushdown** — only reads the columns you actually need
- **Common subplan elimination** — deduplicates repeated work
- **Parallelism** — independent parts of the plan run on separate threads

```
Eager (pl.DataFrame):  reads all → filters → selects  (wasteful)
Lazy  (pl.LazyFrame):  plan → optimise → read only what's needed → execute
```

**Rule of thumb:** Use Lazy API for **any real data pipeline**. Use Eager only for quick exploration or small DataFrames.


In [ ]:
import polars as pl

df = pl.DataFrame({
    "name":       ["Alice", "Bob", "Carol", "Dave", "Eve", "Frank", "Grace"],
    "department": ["Eng", "Eng", "HR", "HR", "Eng", "Finance", "Finance"],
    "salary":     [95000, 82000, 67000, 71000, 110000, 88000, 92000],
    "years":      [5, 3, 8, 2, 10, 6, 4],
})

# --- Convert eager DataFrame to LazyFrame ---
lf = df.lazy()
print("Type:", type(lf))   # polars.LazyFrame

# --- Build a lazy query (nothing executes yet!) ---
query = (
    lf
    .filter(pl.col("salary") > 70_000)
    .with_columns(
        (pl.col("salary") / 12).round(2).alias("monthly"),
        (pl.col("salary") / pl.col("years")).round(0).alias("per_year_exp"),
    )
    .group_by("department")
    .agg(
        pl.col("monthly").mean().round(2).alias("avg_monthly"),
        pl.col("per_year_exp").max().alias("best_per_exp"),
        pl.len().alias("count"),
    )
    .sort("avg_monthly", descending=True)
)

# --- Inspect the optimised plan before executing ---
print("\n=== Optimised query plan ===")
print(query.explain(optimized=True))

# --- Execute with .collect() ---
print("\n=== Result ===")
result = query.collect()
print(result)

# --- scan_csv: lazy reading (never loads entire file into memory) ---
print("\n=== Lazy scan from CSV (no data loaded yet) ===")
# Write a temp CSV to demo scan_csv
df.write_csv("/tmp/employees.csv")
lazy_scan = (
    pl.scan_csv("/tmp/employees.csv")
      .filter(pl.col("department") == "Eng")
      .select("name", "salary")
)
print("Plan built. Executing...")
print(lazy_scan.collect())


---
## Module 10 — I/O: Reading & Writing Files

### Theory
Polars supports high-performance I/O for all common data formats:

| Format | Read | Write | Notes |
|---|---|---|---|
| CSV | `pl.read_csv()` / `pl.scan_csv()` | `.write_csv()` | Universal, but slow & large |
| JSON | `pl.read_json()` / `pl.scan_ndjson()` | `.write_json()` / `.write_ndjson()` | NDJSON = newline-delimited |
| Parquet | `pl.read_parquet()` / `pl.scan_parquet()` | `.write_parquet()` | **Best format**: fast, compressed, typed |
| Excel | `pl.read_excel()` | — | Requires `xlsx2csv` or `openpyxl` |
| Delta Lake | `pl.scan_delta()` | — | Requires `deltalake` package |

**Parquet is the recommended format** for any data pipeline — it preserves dtypes, is heavily compressed, and Polars can read only the columns/rows it needs via predicate pushdown.


In [ ]:
import polars as pl
import tempfile, os

df = pl.DataFrame({
    "name":       ["Alice", "Bob", "Carol", "Dave", "Eve"],
    "department": ["Eng", "Eng", "HR", "HR", "Eng"],
    "salary":     [95000, 82000, 67000, 71000, 110000],
    "years":      [5, 3, 8, 2, 10],
})

tmp = tempfile.mkdtemp()

# --- CSV ---
csv_path = os.path.join(tmp, "employees.csv")
df.write_csv(csv_path)
df_csv = pl.read_csv(csv_path)
print("=== Read from CSV ===")
print(df_csv)
print("dtypes:", df_csv.dtypes)  # note: everything reads as Utf8/Int64

# --- CSV with options ---
df_csv2 = pl.read_csv(csv_path, columns=["name", "salary"], n_rows=3)
print("\n=== CSV: only 2 columns, 3 rows ===")
print(df_csv2)

# --- JSON (newline-delimited) ---
ndjson_path = os.path.join(tmp, "employees.ndjson")
df.write_ndjson(ndjson_path)
df_json = pl.read_ndjson(ndjson_path)
print("\n=== Read from NDJSON ===")
print(df_json)

# --- Parquet: best format, preserves dtypes ---
parquet_path = os.path.join(tmp, "employees.parquet")
df.write_parquet(parquet_path, compression="snappy")
df_parquet = pl.read_parquet(parquet_path)
print("\n=== Read from Parquet ===")
print(df_parquet)
print("dtypes preserved:", df_parquet.dtypes)

# --- Lazy scan_parquet: only reads needed columns from disk ---
print("\n=== scan_parquet (lazy) — only salary column ===")
result = pl.scan_parquet(parquet_path).select("name", "salary").collect()
print(result)

print(f"\nFiles written to: {tmp}")


---
## Module 11 — Handling Missing Data (Nulls)

### Theory
Polars has a **single, consistent null type** — `null` — for all dtypes. This is different from pandas which has confusingly separate `NaN` (float) and `None` (object).

| Operation | Method |
|---|---|
| Count nulls | `.null_count()` |
| Check per row if null | `pl.col("x").is_null()` |
| Check per row if not null | `pl.col("x").is_not_null()` |
| Fill with a value | `.fill_null(value)` |
| Fill with a strategy | `.fill_null(strategy="forward"/"backward"/"mean"/"min"/"max")` |
| Fill with another column | `.fill_null(pl.col("other"))` |
| Drop rows with any null | `.drop_nulls()` |
| Drop rows with null in specific cols | `.drop_nulls(subset=["col"])` |
| Interpolate | `.interpolate()` |


In [ ]:
import polars as pl

df = pl.DataFrame({
    "name":       ["Alice", "Bob", "Carol", "Dave", "Eve"],
    "salary":     [95000, None, 67000, None, 110000],
    "years":      [5, 3, None, 2, 10],
    "department": ["Eng", "Eng", None, "HR", "Eng"],
})

print("=== DataFrame with nulls ===")
print(df)

print("\n=== Null count per column ===")
print(df.null_count())

print("\n=== Rows where salary is null ===")
print(df.filter(pl.col("salary").is_null()))

print("\n=== Fill null salary with column mean ===")
print(df.with_columns(
    pl.col("salary").fill_null(pl.col("salary").mean().round(0))
))

print("\n=== Fill null years with forward fill strategy ===")
print(df.with_columns(
    pl.col("years").fill_null(strategy="forward")
))

print("\n=== Fill null department with literal 'Unknown' ===")
print(df.with_columns(
    pl.col("department").fill_null(pl.lit("Unknown"))
))

print("\n=== Drop rows with ANY null ===")
print(df.drop_nulls())

print("\n=== Drop rows with null only in 'salary' ===")
print(df.drop_nulls(subset=["salary"]))

print("\n=== Interpolate years ===")
print(df.with_columns(pl.col("years").cast(pl.Float64).interpolate()))


---
## Module 12 — Real-World Example: Full EDA Pipeline

### Putting it all together

This module simulates a realistic data analysis pipeline using everything learned above:

1. Generate synthetic sales data
2. Clean nulls
3. Add derived columns
4. Filter and sort
5. Group and aggregate
6. Join with a reference table
7. Use the Lazy API for the full pipeline
8. Write results to Parquet


In [ ]:
import polars as pl
import random, tempfile, os
from datetime import date, timedelta

random.seed(42)

# ── 1. Generate synthetic sales data ──────────────────────────────────────────
n = 200
regions  = ["North", "South", "East", "West"]
products = ["Laptop", "Phone", "Tablet", "Monitor", "Keyboard"]
reps     = [f"Rep_{i:02d}" for i in range(1, 11)]

sales_data = pl.DataFrame({
    "sale_id":    list(range(1, n + 1)),
    "date":       [date(2024, 1, 1) + timedelta(days=random.randint(0, 364)) for _ in range(n)],
    "rep":        [random.choice(reps) for _ in range(n)],
    "region":     [random.choice(regions) for _ in range(n)],
    "product":    [random.choice(products) for _ in range(n)],
    "quantity":   [random.randint(1, 20) for _ in range(n)],
    "unit_price": [round(random.uniform(50, 2000), 2) for _ in range(n)],
    "discount":   [round(random.uniform(0, 0.3), 2) if random.random() > 0.3 else None for _ in range(n)],
})

product_ref = pl.DataFrame({
    "product":   ["Laptop", "Phone", "Tablet", "Monitor", "Keyboard"],
    "category":  ["Computing", "Mobile", "Mobile", "Computing", "Accessories"],
    "cost":      [800.0, 300.0, 250.0, 400.0, 50.0],
})

print(f"Sales records: {sales_data.shape}")
print(sales_data.head(5))
print(f"\nNull count:\n{sales_data.null_count()}")


In [ ]:
import polars as pl, tempfile, os

# Full lazy pipeline: clean → enrich → join → aggregate → report
tmp = tempfile.mkdtemp()
parquet_out = os.path.join(tmp, "sales_summary.parquet")

pipeline = (
    sales_data.lazy()

    # ── 2. Fill missing discounts with 0 ──────────────────────────────────────
    .with_columns(pl.col("discount").fill_null(0.0))

    # ── 3. Add derived columns ─────────────────────────────────────────────────
    .with_columns(
        (pl.col("quantity") * pl.col("unit_price") * (1 - pl.col("discount")))
          .round(2).alias("revenue"),
        pl.col("date").dt.month().alias("month"),
        pl.col("date").dt.quarter().alias("quarter"),
    )

    # ── 4. Join with product reference table ───────────────────────────────────
    .join(product_ref.lazy(), on="product", how="left")

    # ── 5. Add profit column ───────────────────────────────────────────────────
    .with_columns(
        (pl.col("revenue") - pl.col("quantity") * pl.col("cost"))
          .round(2).alias("profit"),
        pl.when(pl.col("revenue") > 5000)
          .then(pl.lit("High"))
          .when(pl.col("revenue") > 1000)
          .then(pl.lit("Mid"))
          .otherwise(pl.lit("Low"))
          .alias("revenue_band"),
    )
)

# ── 6. Group by region + category ─────────────────────────────────────────────
summary = (
    pipeline
    .group_by("region", "category")
    .agg(
        pl.col("revenue").sum().round(2).alias("total_revenue"),
        pl.col("profit").sum().round(2).alias("total_profit"),
        pl.len().alias("num_sales"),
        pl.col("quantity").sum().alias("units_sold"),
        pl.col("rep").n_unique().alias("reps_involved"),
    )
    .sort(["region", "total_revenue"], descending=[False, True])
    .collect()
)

print("=== Revenue & Profit by Region + Category ===")
print(summary)

# ── 7. Top 5 reps by profit ───────────────────────────────────────────────────
top_reps = (
    pipeline
    .group_by("rep")
    .agg(
        pl.col("profit").sum().round(2).alias("total_profit"),
        pl.col("revenue").sum().round(2).alias("total_revenue"),
        pl.len().alias("sales_count"),
    )
    .sort("total_profit", descending=True)
    .head(5)
    .collect()
)

print("\n=== Top 5 Sales Reps by Profit ===")
print(top_reps)

# ── 8. Write results to Parquet ───────────────────────────────────────────────
summary.write_parquet(parquet_out, compression="zstd")
print(f"\n✅ Summary written to: {parquet_out}")
print(f"   File size: {os.path.getsize(parquet_out)} bytes")


---
## Quick Reference & Common Pitfalls

### ✅ Cheat Sheet

```python
import polars as pl

# Create
df = pl.DataFrame({"a": [1,2,3], "b": ["x","y","z"]})
lf = df.lazy()                       # convert to LazyFrame

# Inspect
df.shape                             # (rows, cols)
df.schema                            # column name → dtype mapping
df.describe()                        # statistics
df.head(5) / df.tail(5)

# Select columns
df.select("a", "b")
df.select(pl.col("a") * 2)
df.select(pl.all().exclude("a"))

# Filter rows
df.filter(pl.col("a") > 1)
df.filter((pl.col("a") > 1) & (pl.col("b") == "y"))

# Add/replace columns
df.with_columns(
    (pl.col("a") * 10).alias("a_scaled"),
    pl.when(pl.col("a") > 1).then(pl.lit("big")).otherwise(pl.lit("small")).alias("size")
)

# Sort
df.sort("a", descending=True)
df.sort(["a","b"], descending=[True, False])

# GroupBy
df.group_by("b").agg(pl.col("a").sum(), pl.len())

# Join
df1.join(df2, on="key", how="inner")

# Nulls
df.null_count()
df.fill_null(0)
df.drop_nulls(subset=["a"])

# Lazy pipeline
result = (
    pl.scan_parquet("data.parquet")
      .filter(...)
      .with_columns(...)
      .group_by(...).agg(...)
      .collect()
)

# I/O
df.write_csv("out.csv")
df.write_parquet("out.parquet", compression="zstd")
pl.read_csv("in.csv")
pl.read_parquet("in.parquet")
```

---

### ❌ Common Pitfalls

| Mistake | Problem | Fix |
|---|---|---|
| `df["col"]` for filtering | Returns Series, can't chain | Use `df.filter(pl.col("col") > x)` |
| `df.filter(a > 1 and b == "x")` | `and` doesn't work on expressions | Use `&` and wrap each in `()` |
| Modifying df in place | Polars is immutable | Assign: `df = df.with_columns(...)` |
| Using `df.apply()` / Python loops | Slow — escapes vectorisation | Use expressions, `.map_elements()` only as last resort |
| Forgetting `.collect()` on LazyFrame | Returns a plan, not data | Always call `.collect()` at the end |
| `time.sleep` in group_by | N/A — Polars is always sync | No issue, but don't mix with asyncio unnecessarily |
| Comparing with `== None` | Won't work | Use `pl.col("x").is_null()` |

---

### 🏁 Practice Exercises

1. **Filter & count** — From the sales DataFrame, find all sales in Q4 (months 10–12) with revenue > 3000. How many are there?
2. **Window function** — Add a column showing each sale's revenue as a **percentage of its region's total** using `pl.col("revenue") / pl.col("revenue").sum().over("region")`.
3. **Pivot table** — Use `.pivot()` to create a table where rows = region, columns = product category, values = total revenue.
4. **Lazy pipeline** — Write a full `scan_csv → filter → join → group_by → write_parquet` pipeline without loading the full file into memory.
5. **Chained string ops** — From the employee names, extract the first initial + last name (e.g., "Alice Smith" → "A. Smith") using `.str` methods.
